# {TITLE} — Autoresearch Results

**Dataset:** [{DATASET_NAME}]({DATASET_URL}) ({ROWS} rows)

**Task:** {TASK_TYPE} — predict `{TARGET}` from {FEATURE_DESCRIPTION}

**Approach:** Autonomous ML experiment loop ([autoresearch](https://github.com/detrin/autoresearch)) — {N_EXPERIMENTS} experiments run by an AI agent, logged to MLflow

**Best Result:** {METRIC_NAME} **{BEST_SCORE}** ({BEST_MODEL_DESCRIPTION})

| # | Model | Description | {METRIC_NAME} | Status |
|---|-------|-------------|------|--------|
| 1 | ... | baseline | ... | baseline |
| **N** | **...** | **best** | **...** | **+XX%** |

---
*Generated via [autoresearch](https://github.com/detrin/autoresearch)*

## 1. Setup & Data Loading

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LinearRegression

def find_file(name):
    kaggle_matches = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    if kaggle_matches:
        return kaggle_matches[0]
    if os.path.exists(name):
        return name
    raise FileNotFoundError(f"Cannot find {name}")

RANDOM_STATE = 42
TEST_SIZE = 0.2
TARGET = "{TARGET}"

# df = pd.read_csv(find_file("{DATA_FILE}"))
# print(f"Shape: {df.shape}")
# df.head()

## 2. Exploratory Data Analysis

In [ ]:
print(f"Target stats:\n{df[TARGET].describe()}\n")
print(f"Null counts:\n{df.isnull().sum()[df.isnull().sum() > 0]}")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].hist(df[TARGET], bins=50, edgecolor="black")
axes[0, 0].set_title(f"{TARGET} Distribution")

# axes[0, 1]: mean target by category
# axes[1, 0]: mean target by another category
# axes[1, 1]: scatter of strongest predictor vs target

plt.tight_layout()
plt.show()

## 3. Data Preparation & Feature Engineering

Key features discovered during experiment loop:
- **Feature group 1:** ...
- **Feature group 2:** ...

In [ ]:
train, val = train_test_split(df, test_size=TEST_SIZE, random_state=RANDOM_STATE)

# Feature engineering goes here
# for d in [train, val]:
#     d["new_feature"] = ...

drop_cols = [TARGET]
feature_cols = [c for c in train.columns if c not in drop_cols]

cat_cols = train[feature_cols].select_dtypes(include="object").columns.tolist()
encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    train[col] = le.fit_transform(train[col].astype(str))
    val[col] = le.transform(val[col].astype(str))
    encoders[col] = le

X_train, y_train = train[feature_cols], train[TARGET]
X_val, y_val = val[feature_cols], val[TARGET]

print(f"Train: {X_train.shape}, Val: {X_val.shape}")
print(f"Features ({len(feature_cols)}): {feature_cols[:10]}... (+{len(feature_cols)-10} more)")

## 4. Baseline — Linear Regression

In [ ]:
%%time
lr = LinearRegression()
lr.fit(X_train.fillna(0), y_train)
preds_lr = lr.predict(X_val.fillna(0))
score_lr = mean_absolute_percentage_error(y_val, preds_lr)
print(f"Linear Regression: {score_lr:.6f}")

results = [("LinearRegression", score_lr)]

## 5. Best Model(s) — {MODEL_DESCRIPTION}

In [ ]:
%%time
import lightgbm as lgb

model_lgb = lgb.LGBMRegressor(
    n_estimators=2000, learning_rate=0.05, num_leaves=255, random_state=42, n_jobs=-1,
    min_child_samples=30, subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=0.1,
)
model_lgb.fit(X_train, y_train, eval_set=[(X_val, y_val)],
              callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)])
preds_lgb = model_lgb.predict(X_val)
print(f"LightGBM: {mean_absolute_percentage_error(y_val, preds_lgb):.6f}")

results.append(("LightGBM", mean_absolute_percentage_error(y_val, preds_lgb)))

In [ ]:
%%time
# import xgboost as xgb
# from catboost import CatBoostRegressor
#
# model_xgb = xgb.XGBRegressor(...)
# model_xgb.fit(...)
# preds_xgb = model_xgb.predict(X_val)
#
# model_cb = CatBoostRegressor(...)
# model_cb.fit(...)
# preds_cb = model_cb.predict(X_val)
#
# Print individual scores
# for name, preds in [("XGBoost", preds_xgb), ("CatBoost", preds_cb)]:
#     score = mean_absolute_percentage_error(y_val, preds)
#     print(f"{name}: {score:.6f}")
#     results.append((name, score))

## 6. Weighted Ensemble

In [ ]:
# all_preds = [preds_lgb, preds_xgb, preds_cb]
#
# best_score = 1.0
# best_w = None
# for w1 in np.arange(0.2, 0.8, 0.05):
#     for w2 in np.arange(0.05, 0.6, 0.05):
#         w3 = 1.0 - w1 - w2
#         if w3 < 0.05 or w3 > 0.6:
#             continue
#         blend = w1 * preds_lgb + w2 * preds_xgb + w3 * preds_cb
#         score = mean_absolute_percentage_error(y_val, blend)
#         if score < best_score:
#             best_score = score
#             best_w = (w1, w2, w3)
#
# preds_ensemble = best_w[0] * preds_lgb + best_w[1] * preds_xgb + best_w[2] * preds_cb
# score_ensemble = mean_absolute_percentage_error(y_val, preds_ensemble)
# print(f"Ensemble weights: LGB={best_w[0]:.2f}, XGB={best_w[1]:.2f}, CB={best_w[2]:.2f}")
# print(f"Ensemble: {score_ensemble:.6f}")
#
# results.append(("Ensemble", score_ensemble))

## 7. Feature Importance & Diagnostics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

importance = pd.Series(model_lgb.feature_importances_, index=X_train.columns).sort_values()
importance.tail(15).plot.barh(ax=axes[0])
axes[0].set_title("Top 15 Features (LightGBM split importance)")

sample_idx = np.random.RandomState(42).choice(len(y_val), min(5000, len(y_val)), replace=False)
# Use preds_ensemble or preds_lgb depending on best model
best_preds = preds_lgb  # replace with preds_ensemble if ensemble is best
axes[1].scatter(y_val.iloc[sample_idx], best_preds[sample_idx], alpha=0.15, s=10)
axes[1].plot([y_val.min(), y_val.max()], [y_val.min(), y_val.max()], "r--", alpha=0.5)
axes[1].set_xlabel(f"Actual {TARGET}")
axes[1].set_ylabel(f"Predicted {TARGET}")
axes[1].set_title("Predicted vs Actual")

plt.tight_layout()
plt.show()

## 8. Results Summary

In [ ]:
results_df = pd.DataFrame(results, columns=["Model", "{METRIC_NAME}"]).sort_values("{METRIC_NAME}")
print(results_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 4))
colors = ["#2ecc71" if "Ensemble" in m else "#3498db" for m in results_df["Model"]]
bars = ax.barh(results_df["Model"], results_df["{METRIC_NAME}"], color=colors, edgecolor="black")
ax.set_xlabel("{METRIC_NAME} (lower is better)")
ax.set_title("Model Comparison — {TITLE}")
for bar, val in zip(bars, results_df["{METRIC_NAME}"]):
    ax.text(val + 0.0002, bar.get_y() + bar.get_height() / 2, f"{val:.6f}", va="center")
plt.tight_layout()
plt.show()

## Conclusions

**Best model:** {BEST_MODEL} — **{METRIC_NAME} {BEST_SCORE}** ({IMPROVEMENT}% improvement over baseline)

**What worked:**
- ...

**What didn't work:**
- ...

**Takeaway:** ...

---
*Generated via [autoresearch](https://github.com/detrin/autoresearch) — {N_EXPERIMENTS} autonomous experiments, MLflow-tracked*